# Reading From Bronze Table


In [0]:
%sql
use catalog `workspace`; select * from `silver`.`crm_customers` limit 100;

#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


In [0]:
# renaming columns

RENAME_MAP =(

    ("cst_id", "customer_id"),
    ("cst_key", "customer_key"),
    ("cst_firstname", "firstname"),
    ("cst_lastname", "lastname"),
    ("cst_marital_status", "marital_status"),
    ("cst_gndr", "gender"),
    ("cst_create_date", "created_date")
)


In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

In [0]:
#1. trim the string
#2. Normalization for marital_status, gndr 
#3. Names are not frends

#1. parcours moi toutes les cols, si la colonne est de type tring applique moi un trim
for field in df.schema.fields: 
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
  



In [0]:
# ici on peut constater que les espaces indésirble à droit et a gauche ont été supprimé
df.display()

In [0]:
# Normalization for marital_status, gndr

df =(
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S","Single")
         .when(F.upper(F.col("cst_marital_status")) == "M","Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M","Male")
         .when(F.upper(F.col("cst_gndr")) == "F","Female")
         .otherwise("n/a")
         )

)

In [0]:
df.display()
# 2. Normalization for gender and marital_status done !

In [0]:
# 3.Renaming The columns

for old_name, new_name in RENAME_MAP:
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()
# On constate le renommage des colonnes

# Write Into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customers")
)

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers